# Índice Invertido — Reuters ModApte

Construcción del índice para el corpus Reuters (split ModApte_train).  
El índice almacena, para cada término, los documentos en los que aparece y su frecuencia dentro de cada uno.

In [1]:
import os
import re
import pandas as pd
from collections import Counter, defaultdict
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# nltk.download('stopwords')

## 1. Carga del Corpus

In [2]:
ruta_train = os.path.join(os.getcwd(), 'dataset', 'ModApte_train.csv')

df_reuters = pd.read_csv(ruta_train)
print(f'Documentos en train: {len(df_reuters)}')
print(f'Columnas disponibles: {df_reuters.columns.tolist()}')

Documentos en train: 9603
Columnas disponibles: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']


In [3]:
# Concatenamos título y cuerpo: el título aporta señal limpia que mejora la recuperación
df = pd.DataFrame({
    'id_doc': df_reuters['new_id'],
    'raw': df_reuters['title'].fillna('') + ' ' + df_reuters['text'].fillna('')
})

print(f'Documentos cargados: {len(df)}')
print(df[['id_doc', 'raw']].head(3))

Documentos cargados: 9603
  id_doc                                                raw
0    "1"  BAHIA COCOA REVIEW Showers continued throughou...
1    "5"  NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESER...
2    "6"  ARGENTINE 1986/87 GRAIN/OILSEED REGISTRATIONS ...


## 2. Preprocesamiento

Mismas funciones que en clase, adaptadas al inglés (Reuters es un corpus en inglés).

In [4]:
STOPWORDS = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')

print(f'Stopwords en NLTK: {len(STOPWORDS)} palabras')

Stopwords en NLTK: 198 palabras


In [5]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-z\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def tokenizar(texto, stopwords=STOPWORDS):
    texto_limpio = limpiar_texto(texto)
    tokens = texto_limpio.split()
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
    return tokens

def procesar(texto):
    if not isinstance(texto, str):
        return ""
    tokens = tokenizar(texto)
    stems = [stemmer.stem(t) for t in tokens]
    return " ".join(stems)

In [6]:
df['processed'] = df['raw'].apply(procesar)

print(f'Documentos procesados: {len(df)}')
print(df[['id_doc', 'processed']].head(3))

Documentos procesados: 9603
  id_doc                                          processed
0    "1"  bahia cocoa review shower continu throughout w...
1    "5"  nation averag price farmerown reserv agricultu...
2    "6"  argentin grainoilse registr argentin grain boa...


## 3. Construcción del Índice Invertido

El índice tiene la forma `{término: {id_doc: frecuencia}}`.  
Guardamos la frecuencia por documento porque TF-IDF y BM25 la necesitan después.

In [7]:
# Usamos defaultdict para evitar el chequeo if/else por cada término nuevo
indice_invertido = defaultdict(dict)

for _, fila in df.iterrows():
    id_doc = fila['id_doc']
    frecuencias = Counter(fila['processed'].split())
    for termino, freq in frecuencias.items():
        indice_invertido[termino][id_doc] = freq

print(f'Términos únicos en el índice: {len(indice_invertido):,}')

Términos únicos en el índice: 24,038


## 4. Estadísticas del Índice

In [8]:
df_index = pd.DataFrame([
    {'termino': t, 'df': len(docs), 'tf_total': sum(docs.values())}
    for t, docs in indice_invertido.items()
])

print(f'Términos que aparecen en un solo documento : {(df_index["df"] == 1).sum():,}')
print(f'Términos que aparecen en más de 100 docs   : {(df_index["df"] > 100).sum():,}')
print(f'\nTop 10 términos por document frequency:')
print(df_index.sort_values('df', ascending=False).head(10).to_string(index=False))

Términos que aparecen en un solo documento : 12,133
Términos que aparecen en más de 100 docs   : 884

Top 10 términos por document frequency:
termino   df  tf_total
 reuter 8732      9239
   said 6681     23756
    mln 4599     15385
   dlrs 3988     11042
   year 3514      7182
    pct 3318      9819
    inc 2758      3770
compani 2562      4886
   corp 2395      3242
    cts 2151      5718


## 5. Función de Búsqueda

In [ ]:
def buscar(query, top_k=10):
    terminos = procesar(query).split()
    if not terminos:
        return []

    # Intersección de postings lists: solo docs que contienen TODOS los términos
    postings = [set(indice_invertido.get(t, {}).keys()) for t in terminos]
    docs_relevantes = set.intersection(*postings)

    # Suma de TF como score provisional (se reemplazará por TF-IDF/BM25 después)
    scores = {
        doc_id: sum(indice_invertido[t].get(doc_id, 0) for t in terminos)
        for doc_id in docs_relevantes
    }

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [10]:
# Ejemplos de búsqueda en el índice
queries_ejemplo = ['cocoa trade', 'wheat grain prices', 'oil barrel']

for q in queries_ejemplo:
    resultados = buscar(q)
    print(f"\nQuery: '{q}' → {len(resultados)} documentos (top 5)")
    for doc_id, score in resultados[:5]:
        titulo = df[df['id_doc'] == doc_id]['raw'].values[0][:70]
        print(f"  [{doc_id}] score={score}  {titulo}...")


Query: 'cocoa trade' → 10 documentos (top 5)
  ["5258"] score=30  COCOA LATEST FOCUS FOR COMMODITY PACT NEGOTIATORS The credibility of g...
  ["10760"] score=23  COCOA DEAL SEEN POSITIVE, BUT NO PRICE GUARANTEE The buffer stock rule...
  ["13462"] score=19  COCOA BUFFER STOCK MAY FACE UPHILL BATTLE - TRADE The International Co...
  ["11843"] score=10  MALAYSIA DECLINES TO STATE POSITION ON COCOA PACT Government officials...
  ["11224"] score=9  COMMODITY PACTS MORE ORIENTED TOWARDS MARKET Consuming countries, chas...

Query: 'wheat grain prices' → 10 documentos (top 5)
  ["7326"] score=31  U.S. GRAIN TRADE CALLS SHULTZ REMARK SIGNIFICANT A statement yesterday...
  ["12002"] score=28  SHULTZ USSR TRIP FUELS TALK OF EEP WHEAT OFFER Speculation the United ...
  ["11224"] score=21  COMMODITY PACTS MORE ORIENTED TOWARDS MARKET Consuming countries, chas...
  ["9782"] score=20  IWC LIFTS WORLD GRAIN OUTPUT ESTIMATE TO RECORD The International Whea...
  ["1777"] score=20  MORE SOVIET GRAIN BU

## 6. Recuperación con Similitud Jaccard

Vectores binarios: cada documento se representa como un conjunto de términos (1 si aparece, 0 si no).  
Jaccard mide qué tan parecidos son dos conjuntos: `J(q, d) = |q ∩ d| / |q ∪ d|`

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# binary=True: presencia/ausencia del término, sin contar frecuencia
vectorizador_binario = CountVectorizer(binary=True, lowercase=False)
matriz_binaria = vectorizador_binario.fit_transform(df['processed'])

print(f'Dimensiones de la matriz binaria (Documentos, Vocabulario): {matriz_binaria.shape}')

Dimensiones de la matriz binaria (Documentos, Vocabulario): (9603, 24038)


In [12]:
def recuperar_jaccard(query, top_n=10):
    query_p = procesar(query)
    query_vector = vectorizador_binario.transform([query_p])

    # |q ∩ d|: producto punto entre el vector query y cada documento
    interseccion = matriz_binaria.dot(query_vector.T).toarray().flatten()

    # |q ∪ d| = |q| + |d| - |q ∩ d|
    terminos_por_doc = matriz_binaria.getnnz(axis=1)
    terminos_query   = query_vector.nnz
    union = terminos_por_doc + terminos_query - interseccion

    # np.where evita división por cero cuando query y doc no comparten vocabulario
    similitudes = np.where(union > 0, interseccion / union, 0.0)

    indices_mejores = similitudes.argsort()[-top_n:][::-1]
    resultados = df.iloc[indices_mejores].copy()
    resultados['score_jaccard'] = similitudes[indices_mejores]
    return resultados[['raw', 'score_jaccard']]

In [13]:
queries_ejemplo = ['cocoa trade', 'wheat grain prices', 'oil barrel']

for q in queries_ejemplo:
    res = recuperar_jaccard(q, top_n=1)
    titulo = res['raw'].values[0][:60]
    score  = res['score_jaccard'].values[0]
    print(f"{q:<30} | {titulo:<60} | {score:.4f}")

cocoa trade                    | YEUTTER SAYS U.S. SHOULD STRESS TRADE NEGOTIATIONS AS LONG-T | 0.1250
wheat grain prices             | IWC lifts 1986/87 world wheat, coarse grain estimate one mln | 0.1667
oil barrel                     | CONOCO RAISES CRUDE OIL PRICES UP TO ONE DLR BARREL, WTI AT  | 0.2000


## 7. Recuperación con TF-IDF

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer(lowercase=False)
tfidf_matrix = vectorizador.fit_transform(df['processed'])

print(f"Dimensiones de la matriz (Documentos, Vocabulario): {tfidf_matrix.shape}")

Dimensiones de la matriz (Documentos, Vocabulario): (9603, 24038)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

def recuperar_tfidf(query, top_n=10):
    query_p = procesar(query)
    query_vector = vectorizador.transform([query_p])
    similitudes = cosine_similarity(tfidf_matrix, query_vector).flatten()
    indices_mejores = similitudes.argsort()[-top_n:][::-1]
    resultados = df.iloc[indices_mejores].copy()
    resultados['score_similitud'] = similitudes[indices_mejores]
    
    return resultados[['raw', 'score_similitud']]

In [16]:
queries_ejemplo = ['cocoa trade', 'wheat grain prices', 'oil barrel']

print(f"{'Query del Usuario':<30} | {'Documento Más Relevante':<40} | {'Score'}")

for q in queries_ejemplo:
    res = recuperar_tfidf(q, top_n=1)
    titulo_doc = res['raw'].values[0][:40]
    score = res['score_similitud'].values[0]
    print(f"{q:<30} | {titulo_doc:<40} | {score:.4f}")

Query del Usuario              | Documento Más Relevante                  | Score
cocoa trade                    | ICCO COUNCIL AGREES COCOA BUFFER STOCK R | 0.6110
wheat grain prices             | WORLD GRAIN TRADE RECOVERY MAY BE UNDERW | 0.4767
oil barrel                     | EIA SAYS DISTILLATE, GAS STOCKS OFF IN W | 0.5508


## 8. Recuperación con BM25

In [17]:
from rank_bm25 import BM25Okapi

Adaptar el contenido del DataFrame al formato BM25
Reutilizamos la columna 'processed' que ya tiene el stemming hecho

In [18]:
corpus_para_bm25 = [doc.split() for doc in df['processed']]
bm25 = BM25Okapi(corpus_para_bm25)

In [19]:
def recuperar_bm25(query, top_n=10):
    query_tokens = procesar(query).split()
    puntajes = bm25.get_scores(query_tokens)
    indices_mejores = puntajes.argsort()[-top_n:][::-1]
    resultados = df.iloc[indices_mejores].copy()
    resultados['score_bm25'] = puntajes[indices_mejores]
    
    return resultados[['raw', 'score_bm25']]

In [20]:
for q in queries_ejemplo:
    res = recuperar_bm25(q, top_n=1)
    titulo_doc = res['raw'].values[0][:40]
    score = res['score_bm25'].values[0]
    print(f"{q:<30} | {titulo_doc:<40} | {score:.4f}")

cocoa trade                    | COCOA COUNCIL MEETING ENDS AFTER AGREEIN | 12.1404
wheat grain prices             | WORLD GRAIN TRADE RECOVERY MAY BE UNDERW | 15.4853
oil barrel                     | HAMILTON OIL &lt;HAML> SAYS RESERVES RIS | 11.9685


## 9. Recuperación Semántica

Generamos embeddings con un modelo preentrenado y los almacenamos en FAISS para búsqueda vectorial.  
Usamos el texto `raw` (no `processed`): sentence-transformers necesita lenguaje natural, no stems.

In [25]:
pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   -- ------------------------------------- 1.3/18.9 MB 11.2 MB/s eta 0:00:02
   ------------ --------------------------- 5.8/18.9 MB 18.5 MB/s eta 0:00:01
   ---------------------- ----------------- 10.7/18.9 MB 21.0 MB/s eta 0:00:01
   -------------------------------- ------- 15.5/18.9 MB 22.6 MB/s eta 0:00:01
   ---------------------------------------  18.9/18.9 MB 21.3 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 19.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [26]:
from sentence_transformers import SentenceTransformer
import faiss

In [27]:
modelo = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = modelo.encode(df['raw'].tolist(), show_progress_bar=True)
embeddings = embeddings.astype('float32')

# Normalizar para que el producto punto equivalga a similitud coseno
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatIP(dimension)
indice_faiss.add(embeddings)

print(f'Embeddings generados : {embeddings.shape}')
print(f'Documentos en FAISS  : {indice_faiss.ntotal}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/301 [00:00<?, ?it/s]

Embeddings generados : (9603, 384)
Documentos en FAISS  : 9603


In [28]:
def recuperar_semantico(query, top_n=10):
    query_vector = modelo.encode([query]).astype('float32')
    faiss.normalize_L2(query_vector)
    distancias, indices = indice_faiss.search(query_vector, top_n)
    resultados = df.iloc[indices[0]].copy()
    resultados['score_semantico'] = distancias[0]

    return resultados[['raw', 'score_semantico']]

In [29]:
for q in queries_ejemplo:
    res = recuperar_semantico(q, top_n=1)
    titulo_doc = res['raw'].values[0][:40]
    score = res['score_semantico'].values[0]
    print(f"{q:<30} | {titulo_doc:<40} | {score:.4f}")

cocoa trade                    | COCOA COUNCIL MEETING ENDS AFTER AGREEIN | 0.5967
wheat grain prices             | SMALL QUANTITY OF UK WHEAT SOLD TO HOME  | 0.6431
oil barrel                     | STUDY GROUP URGES INCREASED U.S. OIL RES | 0.4576


## 10. Evaluación

Usamos los topics de Reuters como ground truth: para cada query, los documentos relevantes son todos los que tienen ese topic.

In [30]:
# qrels: {query → conjunto de id_doc relevantes}
# Usamos los topics de Reuters como ground truth
qrels = {
    'cocoa': set(df_reuters[df_reuters['topics'].str.contains('cocoa', na=False)]['new_id']),
    'wheat': set(df_reuters[df_reuters['topics'].str.contains('wheat', na=False)]['new_id']),
    'crude': set(df_reuters[df_reuters['topics'].str.contains('crude', na=False)]['new_id']),
    'earn':  set(df_reuters[df_reuters['topics'].str.contains('earn',  na=False)]['new_id']),
    'trade': set(df_reuters[df_reuters['topics'].str.contains('trade', na=False)]['new_id']),
}

for q, relevantes in qrels.items():
    print(f"'{q}': {len(relevantes)} documentos relevantes")

'cocoa': 55 documentos relevantes
'wheat': 212 documentos relevantes
'crude': 389 documentos relevantes
'earn': 2877 documentos relevantes
'trade': 369 documentos relevantes


In [31]:
def precision_at_k(ids_recuperados, relevantes, k):
    return sum(1 for doc in ids_recuperados[:k] if doc in relevantes) / k

def recall_at_k(ids_recuperados, relevantes, k):
    if not relevantes:
        return 0.0
    return sum(1 for doc in ids_recuperados[:k] if doc in relevantes) / len(relevantes)

def average_precision(ids_recuperados, relevantes):
    if not relevantes:
        return 0.0
    ap, hits = 0.0, 0
    for i, doc in enumerate(ids_recuperados, 1):
        if doc in relevantes:
            hits += 1
            ap += hits / i
    return ap / len(relevantes)

In [32]:
def evaluar(fn_recuperar, nombre, top_n=10):
    filas = []
    aps   = []

    for query, relevantes in qrels.items():
        res = fn_recuperar(query, top_n=top_n)
        # id_doc se recupera usando el índice del DataFrame original
        ids_recuperados = df.loc[res.index, 'id_doc'].tolist()

        p  = precision_at_k(ids_recuperados, relevantes, top_n)
        r  = recall_at_k(ids_recuperados, relevantes, top_n)
        ap = average_precision(ids_recuperados, relevantes)

        filas.append({'query': query, 'precision': round(p, 4), 'recall': round(r, 4), 'AP': round(ap, 4)})
        aps.append(ap)

    df_eval   = pd.DataFrame(filas)
    map_score = sum(aps) / len(aps)

    print(f"\n── {nombre} ──")
    print(df_eval.to_string(index=False))
    print(f"\nMAP: {map_score:.4f}")

    return df_eval, map_score

In [33]:
modelos = [
    (recuperar_jaccard,   'Jaccard'),
    (recuperar_tfidf,     'TF-IDF'),
    (recuperar_bm25,      'BM25'),
    (recuperar_semantico, 'Semántico'),
]

resultados_map = {}
for fn, nombre in modelos:
    _, map_score = evaluar(fn, nombre)
    resultados_map[nombre] = map_score

print("\n── Resumen MAP ──")
for nombre, map_score in resultados_map.items():
    print(f"  {nombre:<12} : {map_score:.4f}")


── Jaccard ──
query  precision  recall     AP
cocoa        1.0  0.1818 0.1818
wheat        0.9  0.0425 0.0402
crude        1.0  0.0257 0.0257
 earn        0.8  0.0028 0.0022
trade        0.6  0.0163 0.0120

MAP: 0.0524

── TF-IDF ──
query  precision  recall     AP
cocoa        1.0  0.1818 0.1818
wheat        1.0  0.0472 0.0472
crude        1.0  0.0257 0.0257
 earn        0.9  0.0031 0.0025
trade        0.9  0.0244 0.0205

MAP: 0.0555

── BM25 ──
query  precision  recall     AP
cocoa        1.0  0.1818 0.1818
wheat        1.0  0.0472 0.0472
crude        1.0  0.0257 0.0257
 earn        0.9  0.0031 0.0025
trade        0.9  0.0244 0.0231

MAP: 0.0560

── Semántico ──
query  precision  recall     AP
cocoa        1.0  0.1818 0.1818
wheat        1.0  0.0472 0.0472
crude        0.9  0.0231 0.0223
 earn        1.0  0.0035 0.0035
trade        0.6  0.0163 0.0099

MAP: 0.0529

── Resumen MAP ──
  Jaccard      : 0.0524
  TF-IDF       : 0.0555
  BM25         : 0.0560
  Semántico    : 0.0529
